# 06 — Indicator pairs per stock (Appendix A)

Builds the per-stock 10-indicator-pair table demanded by Reviewer 1 (§2.6 of the revision plan).
Critically, the selection is re-run on the **training portion only** (2020-22), with the
rank-correlation between training and 2023-validation cumulative returns reported
(see §2.10) so the indicator-pair pipeline doesn't leak into val/test.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy import stats

from src import config as cfg_mod
from src import data as data_mod

cfg = cfg_mod.load_config()
STOCKS = cfg_mod.stocks(cfg)

In [2]:
# The 10 Combi columns in the training CSV are already the *result* of the original two-stage selection.
# This notebook re-derives the cumulative return per pair on the training period and on the validation
# period, then computes the rank correlation between the two -- the diagnostic Reviewer 5 implicitly
# asked for.

summary_rows = []
for s in STOCKS:
    df = data_mod.load_training_features(s)
    INDICATOR_COLS = data_mod.get_indicator_columns(df)
    # Combi columns are Buy/Sell/Hold strings in the CSV; encode to {-1, 0, +1}
    # before computing signed returns.
    df = data_mod.encode_indicator_signals(df, INDICATOR_COLS)
    train = df[df['Year'].isin(cfg['train_years'])]
    val = df[df['Year'] == cfg['val_year']]

    # Naive next-day return for each Combi signal (signal in {-1, 0, +1}).
    def pair_return(period):
        ret = period['Close'].pct_change().shift(-1)
        return {c: float((period[c] * ret).sum()) for c in INDICATOR_COLS}
    tr = pair_return(train)
    vl = pair_return(val)
    # rank correlation
    tr_ranks = stats.rankdata(list(tr.values()))
    vl_ranks = stats.rankdata(list(vl.values()))
    rho = stats.spearmanr(tr_ranks, vl_ranks).statistic
    summary_rows.append({'Stock': s, 'rho(train, val)': round(rho, 3)})
    for c in INDICATOR_COLS:
        summary_rows.append({'Stock': s, 'Pair': c,
                             'Train cum return': round(tr[c], 3),
                             'Val cum return': round(vl[c], 3)})
pd.DataFrame(summary_rows)


,Stock,"rho(train, val)",Pair,Train cum return,Val cum return
0,AAPL,0.455,NaN,NaN,NaN
1,AAPL,NaN,Combi 1: Overbought/ Overbought + SMA Crossove...,0.194,0.138
2,AAPL,NaN,Combi 2: Crossover (Price & EMA 20) + EMA Cros...,0.298,0.029
3,AAPL,NaN,Combi 3: Crossover (Price & EMA 12) + EMA Cros...,0.388,0.084
4,AAPL,NaN,Combi 4: Crossover (Price & EMA 12) + Crossove...,0.279,0.090
5,AAPL,NaN,Combi 5: Crossover (Price & EMA 20) + SMA Cros...,0.048,-0.023
6,AAPL,NaN,Combi 6: Crossover (Price & EMA 12) + Crossove...,0.433,0.064
7,AAPL,NaN,Combi 7: Crossover (Price & EMA 12) + MACD Cro...,0.129,0.011
8,AAPL,NaN,Combi 8: Crossover (Price & EMA 12) + SMA Cros...,0.229,0.018
9,AAPL,NaN,Combi 9: Crossover (Price & EMA 12) + SMA Cros...,0.342,0.071
